# Failure Phase Analysis

Two conditions have a low mean spread $\bar{S}$ yet do not reliably succeed. This notebook checks the phase at timeout in both, and whether the failures are structural or time-limited.

- **(D=10, N=400)**: $\bar{S} = 22.0$, SR = 6% — wall-trapping hypothesis
- **(D=35, N=10)**: $\bar{S} = 21.4$, SR = 70% — overcrowding hypothesis

In [1]:
import pandas as pd

df = pd.read_csv('../data/all_runs_merged_full.csv')
print(f"Total runs: {len(df)}")
print(f"Columns: {list(df.columns)}")

Total runs: 11000
Columns: ['[run number]', 'nb-sheep', 'nb-dogs', 'n-topological', 'W-global-cohesion', 'R-repulsion', 'W-neigh-fleeing-cohesion', 'W-neigh-grazing-cohesion', 'vision-range', 'see-centroid', 'use-density-gradient?', 'with-exit?', '[step]', 'ticks', 'done?', 'current-phase', 'ticks-to-success', 'holding-time', 'sheep-in-zone', 'sheep-captured-count', 'dogs-distance', 'lost-sheep', 'flock-spreadness', 'mean-spreadness']


In [2]:
conditions = [(10, 400), (35, 10)]

for d, n in conditions:
    sub = df[(df['nb-dogs'] == d) & (df['nb-sheep'] == n)]
    success = sub[sub['done?'] == True]
    failed = sub[sub['done?'] == False]
    
    print(f"{'='*60}")
    print(f"  Condition: D={d}, N={n}")
    print(f"{'='*60}")
    print(f"  Total runs:    {len(sub)}")
    print(f"  Successes:     {len(success)} ({100*len(success)/len(sub):.0f}%)")
    print(f"  Failures:      {len(failed)} ({100*len(failed)/len(sub):.0f}%)")
    print()
    
    # --- Failure phase breakdown ---
    phase_counts = failed['current-phase'].value_counts()
    print(f"  Failure breakdown by phase:")
    for phase, count in phase_counts.items():
        pct = 100 * count / len(failed)
        print(f"    {phase:15s}: {count:3d} ({pct:5.1f}%)")
    print()
    
    # --- Could more time help? ---
    n_collecting = phase_counts.get('collecting', 0)
    n_holding = phase_counts.get('holding', 0)
    n_exiting = phase_counts.get('exiting', 0)
    n_temporal = n_holding + n_exiting  # reached later phases → maybe time-limited
    
    print(f"  Structural failures (stuck in collecting): {n_collecting} ({100*n_collecting/len(failed):.1f}%)")
    print(f"  Potentially temporal (holding+exiting):    {n_temporal} ({100*n_temporal/len(failed):.1f}%)")
    print()
    
    # --- Additional stats on failed runs ---
    print(f"  Mean spread (failed):   {failed['mean-spreadness'].mean():.1f}")
    print(f"  Mean spread (success):  {success['mean-spreadness'].mean():.1f}" if len(success) > 0 else "  No successful runs")
    print(f"  Mean ticks (failed):    {failed['ticks'].mean():.0f}")
    print(f"  Mean lost sheep (fail): {failed['lost-sheep'].mean():.1f}")
    print(f"  Mean sheep captured (fail): {failed['sheep-captured-count'].mean():.1f}")
    print(f"  Mean holding time (fail):   {failed['holding-time'].mean():.1f}")
    print()

  Condition: D=10, N=400
  Total runs:    100
  Successes:     6 (6%)
  Failures:      94 (94%)

  Failure breakdown by phase:
    collecting     :  83 ( 88.3%)
    exiting        :   9 (  9.6%)
    holding        :   2 (  2.1%)

  Structural failures (stuck in collecting): 83 (88.3%)
  Potentially temporal (holding+exiting):    11 (11.7%)

  Mean spread (failed):   21.9
  Mean spread (success):  24.2
  Mean ticks (failed):    10000
  Mean lost sheep (fail): 6.3
  Mean sheep captured (fail): 1.3
  Mean holding time (fail):   12.8

  Condition: D=35, N=10
  Total runs:    100
  Successes:     70 (70%)
  Failures:      30 (30%)

  Failure breakdown by phase:
    collecting     :  28 ( 93.3%)
    exiting        :   1 (  3.3%)
    holding        :   1 (  3.3%)

  Structural failures (stuck in collecting): 28 (93.3%)
  Potentially temporal (holding+exiting):    2 (6.7%)

  Mean spread (failed):   19.9
  Mean spread (success):  22.1
  Mean ticks (failed):    10000
  Mean lost sheep (fail): 2

## Exiting failures: sheep already captured

Sheep captured at timeout, for the runs that failed during the exiting phase.

In [3]:
for d, n in conditions:
    sub = df[(df['nb-dogs'] == d) & (df['nb-sheep'] == n)]
    failed_exit = sub[(sub['done?'] == False) & (sub['current-phase'] == 'exiting')]
    
    print(f"D={d}, N={n} — Failures during EXITING: {len(failed_exit)}")
    if len(failed_exit) > 0:
        print(f"  Sheep captured: {failed_exit['sheep-captured-count'].values}")
        print(f"  Lost sheep:     {failed_exit['lost-sheep'].values}")
        print(f"  Holding time:   {failed_exit['holding-time'].values}")
        print(f"  Mean spread:    {failed_exit['mean-spreadness'].values.round(1)}")
        pct_captured = (failed_exit['sheep-captured-count'] / n * 100).values.round(1)
        print(f"  % sheep captured: {pct_captured}")
    print()

D=10, N=400 — Failures during EXITING: 9
  Sheep captured: [  0   0   8   0   0   0   0   0 117]
  Lost sheep:     [ 6  6 10  9  9 12  4  6  0]
  Holding time:   [0 0 0 0 0 0 0 0 0]
  Mean spread:    [20.  21.4 28.4 19.9 19.1 20.4 20.8 22.2 21.8]
  % sheep captured: [ 0.   0.   2.   0.   0.   0.   0.   0.  29.2]

D=35, N=10 — Failures during EXITING: 1
  Sheep captured: [0]
  Lost sheep:     [0]
  Holding time:   [0]
  Mean spread:    [25.3]
  % sheep captured: [0.]



## Summary

Phase breakdown of the failures for the two conditions.

In [4]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print()

for d, n in conditions:
    sub = df[(df['nb-dogs'] == d) & (df['nb-sheep'] == n)]
    failed = sub[sub['done?'] == False]
    phase_counts = failed['current-phase'].value_counts()
    n_fail = len(failed)
    n_collect = phase_counts.get('collecting', 0)
    n_hold = phase_counts.get('holding', 0)
    n_exit = phase_counts.get('exiting', 0)
    
    print(f"--- (D={d}, N={n}) ---")
    print(f"Of the {n_fail} failures, "
          f"{n_collect} ({100*n_collect/n_fail:.0f}%) stall in collecting, "
          f"{n_hold} ({100*n_hold/n_fail:.0f}%) in holding, "
          f"and {n_exit} ({100*n_exit/n_fail:.0f}%) in exiting.")
    
    if n_hold + n_exit > 0:
        print(f"  → {n_hold + n_exit} runs ({100*(n_hold+n_exit)/n_fail:.0f}%) "
              f"reached later phases and might succeed with more time.")
    else:
        print(f"  → All failures are structural (collecting). More time would not help.")
    print()

SUMMARY

--- (D=10, N=400) ---
Of the 94 failures, 83 (88%) stall in collecting, 2 (2%) in holding, and 9 (10%) in exiting.
  → 11 runs (12%) reached later phases and might succeed with more time.

--- (D=35, N=10) ---
Of the 30 failures, 28 (93%) stall in collecting, 1 (3%) in holding, and 1 (3%) in exiting.
  → 2 runs (7%) reached later phases and might succeed with more time.

